[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C45_Privacy_Trustworthy_Course/03_federated_learning/03_federated_learning.ipynb)

# 03 · 联邦学习（用 numpy 从零实现）

目标：把 **FedAvg**、**非 IID 影响**、**客户端采样**、**安全聚合（成对掩码相消）** 用 numpy 实现，并 `assert` 验证。

路线：多客户端数据 → 本地训练 → 加权聚合(对拍集中式) → 非 IID 伤害 → 客户端采样 → 安全聚合 → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊。

> 心智模型：**数据不动、模型动**。FedAvg = 各客户端本地学一会儿 -> 按数据量加权平均 -> 循环。IID 下 ≈ 集中式；非 IID 下被漂移拖累。

## 1 · 多客户端数据 + 本地训练

把数据切给多个客户端，每个客户端在本地跑 SGD。先搭好「本地训练」这块积木（联邦学习的最小单位）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# 玩具线性回归：全局真参数 w_true，各客户端在其本地数据上拟合
D = 5
w_true = np.array([1.0, -2.0, 0.5, 3.0, -1.0])

def make_client_data(n, seed, w=w_true):
    g = np.random.default_rng(seed)
    X = g.standard_normal((n, D))
    y = X @ w + 0.1 * g.standard_normal(n)
    return X, y

def local_train(w_init, X, y, steps=20, lr=0.05):
    '''在本地数据上从 w_init 跑若干步 SGD（全批量梯度），返回本地模型。'''
    w = w_init.copy()
    n = len(y)
    for _ in range(steps):
        grad = X.T @ (X @ w - y) / n        # MSE 梯度
        w -= lr * grad
    return w

X0, y0 = make_client_data(100, seed=1)
w_local = local_train(np.zeros(D), X0, y0, steps=200)
print('单客户端本地训练后 w =', w_local.round(2))
print('真参数            w =', w_true.round(2))
assert np.linalg.norm(w_local - w_true) < 0.2, '单客户端足够数据应能拟合'
print('✅ 本地训练积木就绪：在本地数据上跑 SGD 逼近真参数')

## 2 · FedAvg 聚合：按数据量加权平均

聚合公式：`w = Σ (n_k/n) · w_k`。实现加权平均，并验证它是**数据量加权**（数据多的客户端权重大）。

In [ ]:
def fedavg_aggregate(client_models, client_sizes):
    '''按数据量加权平均各客户端模型。'''
    total = sum(client_sizes)
    agg = np.zeros_like(client_models[0])
    for w_k, n_k in zip(client_models, client_sizes):
        agg += (n_k / total) * w_k
    return agg

# 三个客户端，数据量不同
models = [np.array([0.0,0.0]), np.array([10.0,10.0]), np.array([10.0,10.0])]
sizes = [100, 50, 50]
agg = fedavg_aggregate(models, sizes)
print('加权聚合结果 =', agg, '(client0 权重=0.5，另两个各0.25)')
# 0.5*[0,0] + 0.25*[10,10] + 0.25*[10,10] = [5,5]
assert np.allclose(agg, [5.0, 5.0]), '应按数据量 100:50:50 加权'
# 等数据量时退化为简单平均
agg_eq = fedavg_aggregate([np.array([0.0]), np.array([4.0])], [10, 10])
assert np.allclose(agg_eq, [2.0]), '等数据量 -> 简单平均'
print('✅ FedAvg 聚合：数据量加权正确，等量时退化为简单平均')

## 3 · 对拍集中式：IID + 单步本地，FedAvg == 集中式 SGD

**关键对拍**：当各客户端数据 IID、每个客户端只跑**一步**本地 SGD 时，FedAvg 的聚合更新应**精确等于**在全部数据上做一步集中式 SGD。

这验证了「分布式协作 == 集中训练」在理想情况下成立 —— 联邦学习的正确性基石。

In [ ]:
def one_round_fedavg(w, clients, lr):
    '''各客户端跑一步本地 SGD，再加权聚合。'''
    models, sizes = [], []
    for X, y in clients:
        grad = X.T @ (X @ w - y) / len(y)
        models.append(w - lr * grad)          # 一步本地 SGD
        sizes.append(len(y))
    return fedavg_aggregate(models, sizes)

def centralized_step(w, X_all, y_all, lr):
    grad = X_all.T @ (X_all @ w - y_all) / len(y_all)
    return w - lr * grad

# 同分布(IID)的客户端
clients = [make_client_data(80, seed=10+i) for i in range(4)]
X_all = np.vstack([X for X,_ in clients])
y_all = np.concatenate([y for _,y in clients])

w0 = rng.standard_normal(D)
lr = 0.1
w_fed = one_round_fedavg(w0, clients, lr)
w_cen = centralized_step(w0, X_all, y_all, lr)
print(f'FedAvg vs 集中式 一步后 max|diff| = {np.abs(w_fed - w_cen).max():.2e}')
assert np.allclose(w_fed, w_cen, atol=1e-12), 'IID+单步: FedAvg 应精确=集中式 SGD'
print('✅ 对拍通过：IID + 单步本地时，FedAvg 的聚合更新 == 集中式一步 SGD')

## 4 · 非 IID 的伤害：客户端漂移

现在制造**非 IID**：让每个客户端的数据来自**不同的本地参数**（数据分布偏斜）。多步本地训练后，各客户端跑向各自本地最优，聚合被漂移拖累。对比 IID vs 非 IID 的收敛。

In [ ]:
def run_fedavg(clients, rounds=30, local_steps=10, lr=0.05):
    '''完整 FedAvg：多轮，每轮各客户端多步本地训练后聚合。返回每轮全局损失。'''
    sizes = [len(y) for _, y in clients]
    X_all = np.vstack([X for X,_ in clients]); y_all = np.concatenate([y for _,y in clients])
    w = np.zeros(D)
    losses = []
    for r in range(rounds):
        models = [local_train(w, X, y, steps=local_steps, lr=lr) for X, y in clients]
        w = fedavg_aggregate(models, sizes)
        losses.append(float(np.mean((X_all @ w - y_all)**2)))
    return losses

# IID：所有客户端同一 w_true
iid_clients = [make_client_data(80, seed=20+i, w=w_true) for i in range(5)]
# 非 IID：每个客户端一个被扰动的本地 w（分布偏斜）
noniid_clients = []
for i in range(5):
    w_k = w_true + 2.0 * np.random.default_rng(100+i).standard_normal(D)   # 强异质
    noniid_clients.append(make_client_data(80, seed=30+i, w=w_k))

loss_iid = run_fedavg(iid_clients, local_steps=10)
loss_noniid = run_fedavg(noniid_clients, local_steps=10)
print(f'IID    最终全局损失 = {loss_iid[-1]:.4f}')
print(f'非IID  最终全局损失 = {loss_noniid[-1]:.4f}')
assert loss_iid[-1] < loss_noniid[-1], '非 IID 应收敛更差(漂移拖累)'
print('✅ 非 IID 显著伤害收敛 —— 客户端漂移把全局模型拉扯偏 (联邦学习的命门)')

**本地步数 E 放大漂移**：非 IID 下，本地跑得越多越偏。验证：增大 local_steps 使非 IID 的最终损失更差。

In [ ]:
loss_E2  = run_fedavg(noniid_clients, local_steps=2)
loss_E20 = run_fedavg(noniid_clients, local_steps=20)
print(f'非IID, 本地2步  最终损失 = {loss_E2[-1]:.4f}')
print(f'非IID, 本地20步 最终损失 = {loss_E20[-1]:.4f}')
assert loss_E20[-1] > loss_E2[-1], '非IID下本地步数越多漂移越重、最终越差'
print('✅ 本地步数 E 是「省通信 vs 抗漂移」的旋钮：非IID下E越大漂移越重')

## 5 · 安全聚合：成对掩码相消

让服务器只能解出**总和**、看不到单个更新。技巧：每对客户端 (i,j) 共享随机掩码 `s_ij`，i 加、j 减；求和时两两抵消。

实现并验证：① 单个带掩码更新是「乱码」(远离真实)；② 所有带掩码更新之和 == 真实更新之和。

In [ ]:
def secure_aggregation(updates, seed=0):
    '''成对掩码安全聚合。updates: list of (d,) 向量。返回(带掩码更新列表, 掩码和)。'''
    K = len(updates); d = len(updates[0])
    g = np.random.default_rng(seed)
    # 为每对 (i<j) 生成共享随机掩码
    pair_mask = {}
    for i in range(K):
        for j in range(i+1, K):
            pair_mask[(i,j)] = g.standard_normal(d)
    masked = []
    for i in range(K):
        m = updates[i].copy()
        for j in range(K):
            if i < j: m += pair_mask[(i,j)]     # i<j 加
            elif i > j: m -= pair_mask[(j,i)]    # i>j 减(用同一对掩码)
        masked.append(m)
    return masked

# 5 个客户端的真实更新
true_updates = [np.random.default_rng(200+i).standard_normal(D) for i in range(5)]
masked = secure_aggregation(true_updates, seed=42)

# 单个带掩码更新应是乱码(远离真实)
single_err = np.linalg.norm(masked[0] - true_updates[0])
print(f'单个带掩码更新 vs 真实 距离 = {single_err:.2f}  (应很大 -> 服务器看不出)')
assert single_err > 0.5, '单个带掩码更新应被掩码盖住(乱码)'
# 但所有带掩码更新之和 == 真实之和(掩码两两相消)
sum_masked = np.sum(masked, axis=0)
sum_true = np.sum(true_updates, axis=0)
print(f'带掩码之和 vs 真实之和 max|diff| = {np.abs(sum_masked - sum_true).max():.2e}')
assert np.allclose(sum_masked, sum_true, atol=1e-10), '成对掩码应在求和时相消'
print('✅ 安全聚合：单个更新=乱码(防看单个)，但总和精确正确(掩码相消) —— Bonawitz 2017 内核')

---
## ✏️ 练习 1：FedAvg 聚合（带掉线）

实现 `fedavg_with_dropout(client_models, client_sizes, present)`：只聚合**在线**（present[k]=True）的客户端，按它们的数据量重新归一化加权。掉线客户端不参与。

In [ ]:
def fedavg_with_dropout(client_models, client_sizes, present):
    # TODO: 只对 present[k]=True 的客户端按其数据量加权平均(权重在在线客户端中归一化)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
models = [np.array([0.0]), np.array([4.0]), np.array([10.0])]
sizes = [10, 10, 100]
present = [True, True, False]            # 第3个(大)客户端掉线
agg = fedavg_with_dropout(models, sizes, present)
# 只剩前两个等量客户端 -> 简单平均 = 2.0
assert np.allclose(agg, [2.0]), '应只聚合在线客户端并重新归一化'
# 全在线时等于普通 FedAvg
agg_all = fedavg_with_dropout(models, sizes, [True]*3)
assert np.allclose(agg_all, fedavg_aggregate(models, sizes)), '全在线应=普通FedAvg'
print('✅ 练习 1 通过：会处理客户端掉线的加权聚合')

## ✏️ 练习 2：量化非 IID 程度

实现 `heterogeneity(clients)`：用各客户端**本地最优解**之间的离散程度量化非 IID 程度——对每个客户端解析求最小二乘解 `w_k = lstsq(X_k, y_k)`，返回这些 `w_k` 的**平均两两距离**（越大越异质）。

In [ ]:
def heterogeneity(clients):
    # TODO: 对每个 (X,y) 求 lstsq 本地最优解；返回这些解的平均成对欧氏距离
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
iid_c = [make_client_data(200, seed=40+i, w=w_true) for i in range(4)]
noniid_c = [make_client_data(200, seed=50+i, w=w_true+3*np.random.default_rng(i).standard_normal(D)) for i in range(4)]
h_iid = heterogeneity(iid_c)
h_noniid = heterogeneity(noniid_c)
print(f'IID 异质度={h_iid:.3f}, 非IID 异质度={h_noniid:.3f}')
assert h_noniid > h_iid, '非IID的本地最优应更分散'
assert h_iid < 0.5, 'IID客户端的本地最优应彼此接近'
print('✅ 练习 2 通过：能用本地最优的离散度量化非 IID 程度')

## ✏️ 练习 3：客户端采样与隐私放大

每轮只随机选一部分客户端。实现 `sample_clients(n_clients, sample_rate, rng)`：返回被选中客户端的下标列表（每个客户端独立以 `sample_rate` 概率入选，即 Poisson 采样）。

并理解：被选概率 = sample_rate，正是 DP 子采样放大的 q。

In [ ]:
def sample_clients(n_clients, sample_rate, rng):
    # TODO: 每个客户端独立以 sample_rate 概率入选，返回入选下标的列表
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
rng_t = np.random.default_rng(7)
# 大量重复，平均入选比例应≈sample_rate
counts = [len(sample_clients(100, 0.2, rng_t)) for _ in range(2000)]
mean_selected = np.mean(counts)
print(f'sample_rate=0.2, 平均入选 {mean_selected:.1f}/100 (应≈20)')
assert abs(mean_selected - 20) < 1.5, '平均入选比例应≈sample_rate'
# 返回的是合法下标
sel = sample_clients(100, 0.2, rng_t)
assert all(0 <= i < 100 for i in sel), '下标应合法'
print('✅ 练习 3 通过：Poisson 客户端采样 (sample_rate = DP 子采样放大的 q)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def fedavg_with_dropout(client_models, client_sizes, present):
    idx = [k for k in range(len(present)) if present[k]]
    total = sum(client_sizes[k] for k in idx)
    agg = np.zeros_like(client_models[0])
    for k in idx:
        agg += (client_sizes[k] / total) * client_models[k]
    return agg

In [ ]:
# 练习 2 参考答案
def heterogeneity(clients):
    sols = [np.linalg.lstsq(X, y, rcond=None)[0] for X, y in clients]
    dists = [np.linalg.norm(sols[i]-sols[j])
             for i in range(len(sols)) for j in range(i+1, len(sols))]
    return float(np.mean(dists))

In [ ]:
# 练习 3 参考答案
def sample_clients(n_clients, sample_rate, rng):
    return [k for k in range(n_clients) if rng.random() < sample_rate]

---
## 🧪 真实数据胶囊：Gboard 风格的 cross-device 联邦设置

Google 的 Gboard 输入法是联邦学习最大的真实部署之一：在**海量手机**上联邦训练下一个词预测，配合 client-level DP + 安全聚合。

我们用 Gboard 论文量级的**真实配置**（大量客户端、每轮采样很小比例、每客户端数据很少）跑一个玩具联邦，并验证「客户端采样率 = DP 子采样放大的 q」这条联邦与 DP 的连接。

（配置量级取自公开论文；本环境不联网、用合成数据，关注机制而非真实文本。）

In [ ]:
# Gboard 风格 cross-device 配置（量级真实）
GBOARD = dict(
    total_clients=10000,      # 海量设备
    clients_per_round=100,    # 每轮只采样很小比例
    examples_per_client=20,   # 每台设备数据很少
)
q = GBOARD['clients_per_round'] / GBOARD['total_clients']
print(f"每轮采样率 q = {GBOARD['clients_per_round']}/{GBOARD['total_clients']} = {q}")
print(f'这个 q 同时是 DP 子采样放大的采样率 -> 隐私白送放大约 1/q = {1/q:.0f}x 方向的保护')

# 用这个配置跑玩具联邦：1万客户端太多，抽样模拟其中一批
rng_g = np.random.default_rng(0)
all_clients = [make_client_data(GBOARD['examples_per_client'], seed=1000+i, w=w_true)
               for i in range(200)]    # 模拟 200 个(代表 1 万)
sizes = [len(y) for _, y in all_clients]
X_all = np.vstack([X for X,_ in all_clients]); y_all = np.concatenate([y for _,y in all_clients])
w = np.zeros(D)
for r in range(40):
    sel = [k for k in range(len(all_clients)) if rng_g.random() < 0.5]   # 每轮采样一半
    if not sel: continue
    models = [local_train(w, *all_clients[k], steps=5, lr=0.05) for k in sel]
    w = fedavg_aggregate(models, [sizes[k] for k in sel])
final_loss = float(np.mean((X_all @ w - y_all)**2))
print(f'采样式 FedAvg 40 轮后全局损失 = {final_loss:.4f}')
assert final_loss < 1.0, '采样式联邦应能收敛(IID下)'
print('✅ Gboard 风格配置：每轮只采样部分客户端仍能学，且采样率 q 自带 DP 隐私放大')

**🧪 胶囊练习**：实现 `privacy_amplification_factor(clients_per_round, total_clients)`：返回客户端采样带来的隐私放大因子 `1/q`，并验证「每轮参与的客户端越少（q 越小）-> 放大越强」。

In [ ]:
def privacy_amplification_factor(clients_per_round, total_clients):
    # TODO: 返回 1/q = total_clients / clients_per_round
    raise NotImplementedError

In [ ]:
# 自测
amp_small = privacy_amplification_factor(100, 10000)   # q=0.01
amp_large = privacy_amplification_factor(5000, 10000)  # q=0.5
print(f'每轮100/1万: 放大{amp_small:.0f}x;  每轮5000/1万: 放大{amp_large:.0f}x')
assert amp_small > amp_large, '每轮采样越少 -> 隐私放大越强'
assert abs(amp_small - 100) < 1e-6, '1/0.01=100'
print('✅ 胶囊练习通过：客户端采样越稀疏，DP 隐私放大越强')

In [ ]:
# 📖 胶囊参考答案
def privacy_amplification_factor(clients_per_round, total_clients):
    return total_clients / clients_per_round

### 小结
- **联邦学习 = 数据不动、模型动**：与 DP 正交叠加（联邦防集中，DP 防反推）。
- **FedAvg**：本地多步训练 -> 按数据量加权平均；IID+单步时精确 == 集中式 SGD。
- **非 IID 是命门**：客户端漂移把全局模型拉偏；本地步数 E 与异质度放大漂移。
- **安全聚合**：成对掩码相消让服务器只见总和、看不到单个更新（≠ DP，需叠加）。
- **客户端采样**：省资源 + 增方差 + **自带 DP 隐私放大**（采样率 = q）。

下一站：**模块 04 · 机器遗忘** —— 事后撤回：让模型忘记一个人。